<a href="https://colab.research.google.com/github/phamtuanlinh227-collab/python_for_chemistry/blob/master/Weekend-Projects%20Phase%202/02_dmpnnmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from rdkit import Chem
import pandas as pd
import numpy as np
import deepchem as dc
from deepchem.models.torch_models.dmpnn import DMPNNModel

def sota_gnn_recovery(smiles):
"""
Clean up SMILES and keep only the biggest molecule (Desalting).
We don't want the AI to learn trash noise from random counterions..
"""
  try:
      mol = Chem.MolFromSmiles(smiles)
      if mol is None:
        return None
      # Split fragments (remove salts/water)
      frag = Chem.GetMolFrags(mol, asMols=True) # Convert mol object to frag like complex compound
      if len(frag) > 1:
        largest_frag = max(frag, key=lambda m: m.GetNumAtoms()) # Only extract the largest frag ( inner-sphere ) of the complex
        mol = largest_frag

      if mol.GetBonds() == 0:
        return None
      return Chem.MolToSmiles(mol)
  except Exception as e:
      return None

# Run

# 1. Load raw data
df = pd.read_csv('/content/drive/MyDrive/vscode/chem_master_code/proba_data.csv')

# 2. Clean
df['smiles_clean'] = df['SMILES'].apply(sota_gnn_recovery)
# Isolate valid data from garbage inputs
df_clean = df[df['smiles_clean'].apply(lambda x:isinstance(x, str))]
df_trash = df[~df.index.isin(df_clean.index)]

print("📊 [System Log: Data Pipeline Report]")
print(f"✅ Valid Smiles: {len(df_clean)}")
print(f"❌ Non-Valid Smiles: {len(df_trash)}")

# 3. Featurizer for
smiles_clean = df_clean['smiles_clean'].tolist()
labels_clean = df_clean['BBB_Permeability'].to_numpy()

featurizer = dc.feat.DMPNNFeaturizer()

features = featurizer.featurize(smiles_clean)
# Explicitly convert to numpy array so PyTorch doesn't scream at me later
X_final = np.array(features, dtype=object)
labels_float = np.array(labels_clean).astype(np.float32)
# Initialize DeepChem Dataset Object
dataset = dc.data.NumpyDataset(X=X_final, y=labels_float, ids=smiles_clean)

print("Featurization Complete. Deploying Scaffold Splitter...")

# 4. Use ScaffoldSplitter instead of random split to prevent data leakage
splitter = dc.splits.ScaffoldSplitter()
train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1)

# 5. Initialize the DMPNN Model
model = dc.models.torch_models.DMPNNModel(
    n_tasks = 1,
    mode = 'regression',
    device='cpu',
    dropout=0.2,
    learning_rate=0.001
)

# Let's train AI
model.fit(train_dataset, nb_epoch=50)
print("The AI is done training")

# Check scores
metric = dc.metrics.Metric(dc.metrics.pearson_r2_score)

train_scores = model.evaluate(train_dataset, [metric])
valid_scores = model.evaluate(valid_dataset, [metric])
test_scores = model.evaluate(test_dataset, [metric])

print("=" *40)
print(f"Score of train_dataset: {train_scores['pearson_r2_score']:.4f}")
print(f"Score of valid_dataset: {valid_scores['pearson_r2_score']:.4f}")
print(f"Score of test_dataset: {test_scores['pearson_r2_score']:.4f}")
print("=" * 40)
